# Kaggle Notebook — Stream ZIP dataset without full extraction
Hướng dẫn nhanh: đính kèm dataset qua **Add Data** trong Kaggle Notebook, sau đó dùng đường dẫn `/kaggle/input/...`. Notebook này minh họa cách đọc file trong `archive.zip` từng file một (stream) mà không giải nén toàn bộ.

In [ ]:
# Cell 2: Liệt kê các file trong /kaggle/input để xác định tên dataset và file ZIP
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# Cell 3: Đọc file JSON/JSONL trong ZIP theo dòng (stream), không giải nén toàn bộ
import zipfile
import io
import json

# Thay đường dẫn bên dưới bằng file ZIP thực tế từ /kaggle/input/...
zip_path = '/kaggle/input/your-dataset/archive.zip'

def count_json_objects_in_zip(zip_path):
    total = 0
    with zipfile.ZipFile(zip_path) as z:
        for name in z.namelist():
            if name.endswith('.json') or name.endswith('.jsonl'):
                print('Processing', name)
                with z.open(name) as f:
                    for _line in io.TextIOWrapper(f, encoding='utf-8'):
                        if _line.strip():
                            # mỗi dòng là một JSON object (adjust if file is a JSON array)
                            total += 1
    return total

print('Total JSON objects (approx):', count_json_objects_in_zip(zip_path))

In [ ]:
# Cell 4: Đọc CSV bên trong ZIP theo chunk với pandas (không tải toàn bộ vào RAM)
import pandas as pd
import zipfile, io

zip_path = '/kaggle/input/your-dataset/archive.zip'
with zipfile.ZipFile(zip_path) as z:
    for name in z.namelist():
        if name.endswith('.csv'):
            print('Reading CSV:', name)
            with z.open(name) as f:
                reader = pd.read_csv(io.TextIOWrapper(f, encoding='utf-8'), chunksize=100000)
                for i, chunk in enumerate(reader, start=1):
                    print(f'  chunk #{i} rows={len(chunk)}')
                    # xử lý từng chunk ở đây (ví dụ: tính, lọc, gửi xuống DB, etc.)

## Cách đính kèm dataset trong Kaggle Notebook
- Mở notebook trên Kaggle.
- Bên phải, click **Add data** → **Datasets** → chọn dataset có sẵn hoặc upload file ZIP của bạn.
- Sau khi đính kèm, file sẽ nằm dưới `/kaggle/input/<dataset-name>/` và bạn dùng đường dẫn đó trong các cell trên.

Lưu ý: khi bạn attach dataset, Kaggle cung cấp nó read-only trong `/kaggle/input`. Ghi kết quả hoặc artifacts vào `/kaggle/working` để tải về from the Notebook 
 tab.

## Gợi ý thêm
- Nếu dataset quá lớn để lưu trong Kaggle, tải dataset vào một cloud storage (GCS/S3) và đọc trực tiếp từ đó.
- Bạn không cần `kaggle.json` khi dùng dataset đã được **Add data**; chỉ cần `kaggle` API nếu muốn download programmatically từ API vào môi trường khác.
- Muốn mình tùy chỉnh notebook này cho dataset cụ thể của bạn không (hãy gửi tên dataset hoặc đường dẫn trong `/kaggle/input`) ?